# CLIMP reproduction (CS613) — Kaggle runner
All logic lives in `src/`; this notebook installs, runs two GPU queues in parallel, and zips the results.
Settings: **Accelerator = GPU T4 x2, Internet = On**, secret **`HF_TOKEN`** attached.
Set `MODE` in cell 2 (`"smoke"` first, then `"full"`), then **Save Version → Save & Run All**.
Download `/kaggle/working/results_<MODE>.zip` from the Output tab.

In [ ]:
!nvidia-smi
!df -h /tmp /kaggle/working | cat
!free -g

In [ ]:
import os, subprocess, sys, threading, time
from pathlib import Path

MODE = "smoke"   # "smoke" (~25 min) or "full" (~1.5-2 h)
E8_EPOCHS = 20   # CIFAR epochs for the full run (chosen from the smoke timing)

REPO_URL = "https://github.com/tanishwanve-git/climp-cs613.git"
UPSTREAM_COMMIT = "3ff1279bb87f905596e6187f0e4a14f2cd28e818"
REPO = Path("/kaggle/working/climp-cs613")
LOGS = Path("/kaggle/working/logs")
os.environ["DATA_DIR"] = "/tmp/data"            # datasets + HF cache stay out of /kaggle/working (20 GB limit)
os.environ["HF_HOME"] = "/tmp/data/hf"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret (downloads may be rate-limited):", e)

In [ ]:
t0 = time.time()
!rm -rf {REPO} && git clone -q {REPO_URL} {REPO}
!git clone -q https://github.com/NimrodShabtay/CLIMP.git {REPO}/CLIMP && git -C {REPO}/CLIMP checkout -q {UPSTREAM_COMMIT}
!echo "repo: $(git -C {REPO} rev-parse --short HEAD)  upstream: $(git -C {REPO}/CLIMP rev-parse --short HEAD)"

# 1) torch 2.10 (CUDA 12.6): the newest torch with official prebuilt mamba-ssm / causal-conv1d wheels.
#    Kaggle's newer torch has none, which forces a 40-60 min compile. torchaudio must match torch, because
#    transformers imports it if it is installed.
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126
# 2) prebuilt CUDA kernels for torch 2.10 + CUDA 12 + this Python
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEELS = [
    f"https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.10cxx11abiTRUE-{PY}-{PY}-linux_x86_64.whl",
    f"https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-{PY}-{PY}-linux_x86_64.whl",
]
!pip install -q {" ".join(WHEELS)}
# 3) everything else: upstream requirements (minus torch / kernels) + ours
!grep -viE "^(torch|torchvision|triton|causal-conv1d|mamba-ssm)" {REPO}/CLIMP/requirements.txt > /tmp/upstream_reqs.txt
!pip install -q -r /tmp/upstream_reqs.txt -r {REPO}/requirements.txt

# 4) self-test: the kernels must actually run on the T4
Path("/tmp/check_kernels.py").write_text('''
import torch, transformers, mamba_ssm, causal_conv1d
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from causal_conv1d import causal_conv1d_fn
import selective_scan_cuda
d = "cuda"
y = selective_scan_fn(torch.randn(1, 4, 8, device=d), torch.rand(1, 4, 8, device=d), -torch.rand(4, 1, device=d),
                      torch.randn(1, 1, 8, device=d), torch.randn(1, 1, 8, device=d))
z = causal_conv1d_fn(torch.randn(1, 8, 16, device=d), torch.randn(8, 4, device=d))
torch.cuda.synchronize()
print("kernels OK on", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| mamba_ssm", mamba_ssm.__version__, "| causal_conv1d", causal_conv1d.__version__)
''')
!python /tmp/check_kernels.py
if _exit_code != 0:
    print("!!! prebuilt kernels failed -> compiling from source (slow, 40-60 min)")
    !pip install -q causal-conv1d mamba-ssm --no-build-isolation
    !python /tmp/check_kernels.py
# 5) every library the experiment scripts import must load; stop here (not 20 min later) if one does not
Path("/tmp/check_imports.py").write_text('''
import torch, torchvision, transformers, open_clip, timm, webdataset, fvcore, scipy, matplotlib, pyarrow
from transformers import AutoTokenizer, MambaForCausalLM, Mamba2ForCausalLM
import sys; sys.path.insert(0, "CLIMP")
from models import load_climp, build_vmamba, MambaCLIP
print("imports OK | torchvision", torchvision.__version__, "| transformers", transformers.__version__, "| open_clip", open_clip.__version__)
''')
!cd {REPO} && python /tmp/check_imports.py
if _exit_code != 0:  # transformers imports torchaudio whenever it is installed; this project never uses it
    print("!!! import check failed -> uninstalling torchaudio and retrying")
    !pip uninstall -y -q torchaudio
    !cd {REPO} && python /tmp/check_imports.py
assert _exit_code == 0, "an import failed: fix the environment before running experiments"
print(f"install total: {time.time() - t0:.0f}s")

In [ ]:
import psutil

OUT = REPO / ("results_smoke" if MODE == "smoke" else "results")
S = " --smoke" if MODE == "smoke" else ""
# (name, command, uses CLIMP weights -> only runs if the E1 README gate passed)
if MODE == "smoke":
    QUEUES = {
        0: [("pytest", "python -m pytest tests -q", False),
            ("E1", "python src/exp_nocaps.py --smoke", False),
            ("E2", "python src/exp_geometry.py --smoke", False),
            ("E3_E4", "python src/exp_zeroshot.py --smoke --shuffle-grid 1 4", False)],
        1: [("E6", "python src/exp_efficiency.py --smoke", False),
            ("E8", "python src/exp_cifar_bias.py --smoke", False),
            ("E5", "python src/exp_docci.py --smoke", False),
            ("E7", "python src/exp_sugarcrepe.py --smoke", False)],
    }
else:
    QUEUES = {
        0: [("E1", "python src/exp_nocaps.py", False),
            ("E2", "python src/exp_geometry.py", False),
            ("E3", "python src/exp_zeroshot.py", True),
            ("E4", "python src/exp_zeroshot.py --datasets imagenetv2 --shuffle-grid 2 4 8", True)],
        1: [("E6", "python src/exp_efficiency.py", False),
            ("E8", f"python src/exp_cifar_bias.py --epochs {E8_EPOCHS}", False),
            ("E5", "python src/exp_docci.py", True),
            ("E7", "python src/exp_sugarcrepe.py", True)],
    }

TIMEOUT_MIN = 25 if MODE == "smoke" else 240  # a stuck job is killed so its queue moves on
LOGS.mkdir(exist_ok=True)
status, lock, e1_done = [], threading.Lock(), threading.Event()
say = lambda msg: (lock.acquire(), print(msg, flush=True), lock.release())

def tail(path, n=15):
    return "\n".join(path.read_text(errors="replace").splitlines()[-n:])

def run_queue(gpu, jobs):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    for name, cmd, climp in jobs:
        if climp:  # wait until CLIMP-Mamba1 @224 has been checked against the README, then obey the gate
            while not ((OUT / "nocaps_climp-mamba1_224.json").exists() or e1_done.is_set()):
                time.sleep(20)
            if (OUT / "GATE_FAILED").exists() or not (OUT / "nocaps_climp-mamba1_224.json").exists():
                status.append((name, gpu, "SKIPPED (E1 gate failed or E1 crashed)", 0))
                say(f"[GPU{gpu}] skip {name}: E1 gate not passed")
                continue
        say(f"[GPU{gpu}] start {name}: {cmd}")
        t = time.time()
        log = LOGS / f"{name}.log"
        with open(log, "w") as f:
            try:  # "exec" makes the job the shell process itself, so the timeout kills the job, not just the shell
                code = subprocess.run("exec " + cmd, shell=True, cwd=REPO, env=env, stdout=f, stderr=subprocess.STDOUT,
                                      timeout=TIMEOUT_MIN * 60).returncode
            except subprocess.TimeoutExpired:
                code = f"timeout after {TIMEOUT_MIN} min"
        if name == "E1":
            e1_done.set()
        status.append((name, gpu, "ok" if code == 0 else f"FAILED ({code})", time.time() - t))
        say(f"[GPU{gpu}] {name} {'ok' if code == 0 else 'FAILED'} in {(time.time() - t) / 60:.1f} min. Last lines:\n{tail(log)}\n")

threads = [threading.Thread(target=run_queue, args=(g, jobs)) for g, jobs in QUEUES.items()]
[t.start() for t in threads]
start, peak, last = time.time(), 0.0, 0.0
while any(t.is_alive() for t in threads):
    used = psutil.virtual_memory().used / 1e9
    peak = max(peak, used)
    if time.time() - last > 300:  # heartbeat every 5 min
        gpu_mem = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader", shell=True,
                                 capture_output=True, text=True).stdout.split()
        say(f"[{(time.time() - start) / 60:.0f} min] RAM {used:.1f} GB (peak {peak:.1f} of "
            f"{psutil.virtual_memory().total / 1e9:.0f}) | GPU memory: {' / '.join(gpu_mem[0::2])} MiB")
        last = time.time()
    time.sleep(10)
[t.join() for t in threads]
print(f"\nall queues finished in {(time.time() - start) / 60:.1f} min, peak RAM {peak:.1f} GB")
for name, gpu, st, secs in status:
    print(f"  GPU{gpu}  {name:7s} {st:40s} {secs / 60:6.1f} min")

In [ ]:
!cd {REPO} && python src/make_results.py {S}
name = f"results_{MODE}"
!rm -f /kaggle/working/{name}.zip && cd {REPO} && cp -r {LOGS} {OUT}/logs && zip -qr /kaggle/working/{name}.zip {OUT.name} -x "*/emb/*" && ls -lh /kaggle/working/{name}.zip